# E29 - LightGBM + gain SELECTION (winners' toolchain)

The 2nd-place pipeline was *large bank -> SHAP/gain selection -> LightGBM* (+TabPFN). Our
E25 approximates the bank+selection idea with ExtraTrees/HistGBM; this notebook isolates the
**model + selection** lever with the winners' actual tool.

- **Requires LightGBM** (not in the base env): `pip install lightgbm`. The first cell guards
  the import and sets `HAS_LGBM`.
- Comparison A (model swap): champion features, **LGBM vs champion HGB** - isolates the model.
- Comparison B (bank + select): champion features + cheap extra families (Hurst, AR(4)) ->
  LGBM **gain** importance -> keep gain>0 -> LGBM on the kept subset.

Hardened bar: `t >= 2.5` vs champion. `SMOKE=1` for a quick check.

In [ ]:
import os, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print("lightgbm", lgb.__version__)
except Exception as e:
    HAS_LGBM = False
    print("LightGBM NOT installed -> run:  .venv/bin/pip install lightgbm")
    print("   (import error:", e, ")")

from _overnight import (
    load_pool, select_picks, evaluate, paired, build_training, fold_auc,
    SMOKE, BASE, N_JOBS,
)
from _exp_fast import split_series
print("SMOKE =", SMOKE)

from _feat_lib import _feat

BASE_KEYS = ("spec_entropy",)   # base15 builder
SCALES = (60, 130, 250)         # champion multiscale
WINDOW = 250


def _subwin(w, s):
    return w if s >= len(w) else w[-s:]


def champion_feat(w, ctx):
    out = []
    for s in SCALES:
        out.extend(_feat(_subwin(w, s), ctx, keys=BASE_KEYS))
    return out

def verdict(md_, t):
    if md_ > 0 and t >= 2.5:
        return "SHIP(hard)"
    if md_ > 0 and t >= 2.0:
        return "marginal(HOLD)"
    if md_ > 0 and t >= 1.5:
        return "weak"
    return "noise"


In [ ]:
# Cheap extra families for the bank (ranked high in E25 smoke): Hurst + AR(4).
EPS = 1e-12
AR_ORDER = 4


def _hurst_rs(x):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 16:
        return 0.5
    y = x - x.mean()
    z = np.cumsum(y)
    R = float(z.max() - z.min())
    S = float(x.std())
    if S < EPS or R < EPS:
        return 0.5
    return math.log(R / S) / math.log(n)


def _ar_coefs(x, p=AR_ORDER):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    n = len(x)
    if n <= p + 1:
        return [0.0] * p
    r = np.array([float(np.dot(x[:n - k], x[k:])) / n for k in range(p + 1)])
    if r[0] < EPS:
        return [0.0] * p
    R = np.empty((p, p))
    for i in range(p):
        for j in range(p):
            R[i, j] = r[abs(i - j)]
    try:
        phi = np.linalg.solve(R, r[1:p + 1])
    except np.linalg.LinAlgError:
        phi = np.zeros(p)
    return [float(v) for v in phi]


def bank_feat(w, ctx):
    out = champion_feat(w, ctx)   # 45 base
    for s in SCALES:
        ws = _subwin(w, s)
        out.append(_hurst_rs(ws))
        out.extend(_ar_coefs(ws))
    return out


def make_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        reg_lambda=1.0, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, n_jobs=1, random_state=42, verbose=-1,
    )


In [ ]:
if not HAS_LGBM:
    raise SystemExit("Install lightgbm first: .venv/bin/pip install lightgbm")

X, yi, y_val, train_ids, folds = load_pool()
picks = select_picks(X, train_ids)

# champion baseline (HGB on champion features)
base_aucs, nf = evaluate(champion_feat, WINDOW, X, yi, y_val, picks, folds)
print(f"champion HGB  nf={nf} mean={base_aucs.mean():.4f}\n")


def eval_lgbm(feat_fn, keep_mask=None):
    Xtr, ytr = build_training(X, yi, picks, feat_fn, WINDOW)
    if keep_mask is not None:
        Xtr = Xtr[:, keep_mask]
        fn = lambda w, ctx: list(np.asarray(feat_fn(w, ctx))[keep_mask])
    else:
        fn = feat_fn
    model = make_lgbm()
    model.fit(Xtr, ytr)
    aucs = np.asarray([fold_auc(model, fn, WINDOW, X, y_val, f) for f in folds])
    return aucs, Xtr.shape[1]


# A) model swap on champion features
a, na = eval_lgbm(champion_feat)
md_, sd, t, _ = paired(a, base_aucs)
print(f"A  LGBM champion-feats  nf={na} mean={a.mean():.4f}  delta={md_:+.4f}  t={t:+.2f}  -> {verdict(md_, t)}")

# B) bank + gain selection
Xtr, ytr = build_training(X, yi, picks, bank_feat, WINDOW)
sel = make_lgbm().fit(Xtr, ytr)
gain = sel.booster_.feature_importance(importance_type="gain")
keep_mask = gain > 0
print(f"   bank nf={Xtr.shape[1]} -> kept {int(keep_mask.sum())} by gain>0")
b, nb = eval_lgbm(bank_feat, keep_mask)
md2, sd2, t2, _ = paired(b, base_aucs)
print(f"B  LGBM bank+select     nf={nb} mean={b.mean():.4f}  delta={md2:+.4f}  t={t2:+.2f}  -> {verdict(md2, t2)}")


## Interpretation

- **A** isolates the model: if LGBM beats HGB on identical features at t>=2.5, the champion's
  ceiling was the model, not the features - swap it (note LGBM must be available in the crunch
  submission env; check before shipping).
- **B** is the full winners' recipe (bank -> gain-select -> LGBM). If it clears t>=2.5, wire the
  kept features + LGBM into a submission notebook and local-test.

If both are noise, the 7pt gap to the leaders is confirmed to be the *problem structure*
(2025 known-boundary vs our real-time unknown-boundary), not the model/toolchain - bank the
champion and stop chasing last year's pipeline.